# NLP Lab 02 Tasks

## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

In [ ]:
# (a)
from PyPDF2 import PdfReader
import re
from collections import Counter

pdf_path = "sample.pdf"
reader = PdfReader(pdf_path)
pages = [page.extract_text() or "" for page in reader.pages]
print("Total pages:", len(pages))

# (b)
all_text = " ".join(pages)
tokens = re.sub(r"[^A-Za-z\s]", " ", all_text).lower().split()
print("Total tokens:", len(tokens))

# (c)
stopwords = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "he", "in", "is", "it", "its", "of", "on", "that", "the", "to", "was", "were", "will", "with"}
word_freq = Counter(word for word in tokens if word not in stopwords)
for word, count in word_freq.most_common(10):
    print(word, count)

# (d)
keyword = input("Enter keyword: ").strip().lower()
found = False
for page_number, page_text in enumerate(pages, start=1):
    for sentence in re.split(r"(?<=[.!?])\s+", page_text):
        if keyword in sentence.lower():
            print(f"Page {page_number}: {sentence.strip()}")
            found = True
if not found:
    print("No matching sentence found.")

## Task 2: Structured Page Metadata Extraction and CSV Export

In [ ]:
# (a)
from PyPDF2 import PdfReader
import csv

pdf_path = "sample.pdf"
reader = PdfReader(pdf_path)
pages_data = []

for page_number, page in enumerate(reader.pages, start=1):
    raw_text = page.extract_text() or ""
    pages_data.append({
        "page_number": page_number,
        "raw_text": raw_text,
        "word_count": len(raw_text.split()),
        "char_count": len(raw_text.replace(" ", ""))
    })
print(pages_data[:2])

# (b)
highest = max(pages_data, key=lambda page: page["word_count"])
non_empty = [page for page in pages_data if page["word_count"] > 0]
lowest = min(non_empty, key=lambda page: page["word_count"]) if non_empty else None
average = sum(page["word_count"] for page in pages_data) / len(pages_data)

print("Highest word-count page:", highest["page_number"])
print(highest["raw_text"])
print("Lowest non-zero word-count page:", lowest["page_number"] if lowest else "None")
print("Average word count:", round(average, 2))

# (c)
class PhraseNotFoundError(Exception):
    pass

phrase = input("Enter search phrase: ").strip().lower()
try:
    matches = [(page["page_number"], page["raw_text"].lower().count(phrase)) for page in pages_data if phrase in page["raw_text"].lower()]
    if not matches:
        raise PhraseNotFoundError(f"'{phrase}' was not found.")
    for page_number, count in matches:
        print(f"Page {page_number}: {count} occurrence(s)")
except PhraseNotFoundError as error:
    print(error)

# (d)
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["page_number", "word_count", "char_count"])
    writer.writeheader()
    writer.writerows({key: page[key] for key in writer.fieldnames} for page in pages_data)

with open("pdf_summary.csv", newline="", encoding="utf-8") as file:
    for row in csv.reader(file):
        print(row)

## Task 3: PDF Text Preprocessing Pipeline for NLP

In [ ]:
# (a)
from PyPDF2 import PdfReader
from nltk.corpus import stopwords
import nltk
import re

nltk.download("stopwords", quiet=True)
pdf_path = "sample.pdf"
reader = PdfReader(pdf_path)
text = " ".join(page.extract_text() or "" for page in reader.pages)
before_count = len(text)
cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", text)
cleaned_text = re.sub(r"\d+", " ", cleaned_text)
cleaned_text = re.sub(r"[^A-Za-z.\s]", " ", cleaned_text)
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
print("Before cleaning:", before_count)
print("After cleaning:", len(cleaned_text))

# (b)
raw_sentences = re.split(r"[.!?]", cleaned_text.lower())
sentences = [sentence.strip() for sentence in raw_sentences if sentence.strip()]
print("Before removing empty strings:", len(raw_sentences))
print("After removing empty strings:", len(sentences))

# (c)
english_stopwords = set(stopwords.words("english"))
processed_sentences = [" ".join(word for word in sentence.split() if word not in english_stopwords) for sentence in sentences]
filtered_sentences = [sentence for sentence in processed_sentences if len(sentence.split()) >= 5]
print("Before filtering:", len(processed_sentences))
print("After filtering:", len(filtered_sentences))
for sentence in filtered_sentences[:3]:
    print(sentence)

# (d)
with open("nlp_ready_output.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(filtered_sentences))

with open("nlp_ready_output.txt", encoding="utf-8") as file:
    lines = [line.strip() for line in file if line.strip()]

print("Total lines:", len(lines))
print("Longest sentence:", max(lines, key=len) if lines else "None")
print("Shortest sentence:", min(lines, key=len) if lines else "None")

## Task 4: DOCX Paragraph Classification and Statistical Analysis

In [ ]:
# (a)
from docx import Document
from pprint import pprint

docx_path = "sample.docx"
doc = Document(docx_path)
paragraphs_data = [{"text": paragraph.text, "style": paragraph.style.name} for paragraph in doc.paragraphs]
print("Total paragraphs:", len(paragraphs_data))
print("Unique styles:", sorted({item["style"] for item in paragraphs_data}))

# (b)
groups = {"Headings": [], "Body": [], "Other/Empty": []}
for item in paragraphs_data:
    if not item["text"].strip():
        groups["Other/Empty"].append(item)
    elif item["style"].startswith("Heading"):
        groups["Headings"].append(item)
    elif item["style"] == "Normal" or "Body" in item["style"]:
        groups["Body"].append(item)
    else:
        groups["Other/Empty"].append(item)
for group, items in groups.items():
    print(group, len(items))

# (c)
body_stats = [{"text": item["text"], "word_count": len(item["text"].split()), "char_count": len(item["text"].replace(" ", ""))} for item in groups["Body"] if len(item["text"].split()) > 5]
if body_stats:
    highest = max(body_stats, key=lambda item: item["word_count"])
    lowest = min(body_stats, key=lambda item: item["word_count"])
    mean = sum(item["word_count"] for item in body_stats) / len(body_stats)
    print("Highest:", highest)
    print("Lowest above 5 words:", lowest)
    print("Mean word count:", round(mean, 2))
else:
    print("No qualifying body paragraphs.")

# (d)
style_data = {}
for item in paragraphs_data:
    style_data.setdefault(item["style"], {"count": 0, "texts": []})
    style_data[item["style"]]["count"] += 1
    style_data[item["style"]]["texts"].append(item["text"])
for data in style_data.values():
    if data["count"] > 2:
        data["texts"] = data["texts"][:2]
style_data = dict(sorted(style_data.items(), key=lambda item: item[1]["count"], reverse=True))
pprint(style_data)

## Task 5: DOCX Table Extraction and Pandas Integration

In [ ]:
# (a)
from docx import Document
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph
import pandas as pd

docx_path = "document_with_table.docx"
doc = Document(docx_path)
tables_data = [[[cell.text for cell in row.cells] for row in table.rows] for table in doc.tables]
print("Total tables:", len(tables_data))
for number, table in enumerate(tables_data, start=1):
    print(f"Table {number}: {len(table)} x {len(table[0]) if table else 0}")

# (b)
if not tables_data:
    raise ValueError("The document contains no tables.")
first_table = tables_data[0]
df = pd.DataFrame(first_table[1:], columns=first_table[0])
print(df.to_string(index=False))
print(df.replace("", pd.NA).isna().sum())

# (c)
pre_table_paragraphs = []
for child in doc.element.body.iterchildren():
    if child.tag == qn("w:tbl"):
        break
    if child.tag == qn("w:p"):
        pre_table_paragraphs.append(Paragraph(child, doc).text)
print("Pre-table paragraph count:", len(pre_table_paragraphs))

# (d)
with open("docx_extracted.txt", "w", encoding="utf-8") as file:
    file.write("--- PRE-TABLE PARAGRAPHS ---\n")
    file.write("\n".join(pre_table_paragraphs))
    file.write("\n\n--- TABLE DATA ---\n")
    file.write("\n".join(" | ".join(row) for row in first_table))

## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

In [ ]:
# (a)
from docx import Document
from collections import Counter
import re

doc_a = Document("document_a.docx")
doc_b = Document("document_b.docx")
headings_a = [p.text.strip() for p in doc_a.paragraphs if p.style.name.startswith("Heading") and p.text.strip()]
headings_b = [p.text.strip() for p in doc_b.paragraphs if p.style.name.startswith("Heading") and p.text.strip()]
print("Common headings:", set(headings_a) & set(headings_b))
print("Only in Document A:", set(headings_a) - set(headings_b))
print("Only in Document B:", set(headings_b) - set(headings_a))

# (b)
body_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip()]
body_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip()]
merged = body_a + body_b
deduplicated = list({paragraph.strip().lower(): paragraph.strip() for paragraph in merged}.values())
print("Before deduplication:", len(merged))
print("After deduplication:", len(deduplicated))

# (c)
stopwords = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "that", "the", "to", "was", "were", "with"}
paragraph_tokens = [re.sub(r"[^a-z\s]", " ", paragraph.lower()).split() for paragraph in deduplicated]
all_tokens = [word for tokens in paragraph_tokens for word in tokens if word not in stopwords]
frequency = Counter(all_tokens)
print(frequency.most_common(20))

# (d)
lengths = [len(tokens) for tokens in paragraph_tokens]
longest_index = max(range(len(lengths)), key=lengths.__getitem__) if lengths else None
summary = {
    "total_headings": len(headings_a) + len(headings_b),
    "unique_body_paragraphs": len(deduplicated),
    "vocabulary_size": len(set(all_tokens)),
    "average_paragraph_length": round(sum(lengths) / len(lengths), 2) if lengths else 0,
    "longest_paragraph": deduplicated[longest_index] if longest_index is not None else "None"
}
for key, value in summary.items():
    print(f"{key}: {value}")

## Task 7: Multi-Endpoint API Data Joining and Analysis

In [ ]:
# (a)
import requests
from pprint import pprint

users_response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10)
posts_response = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=10)
if users_response.status_code != 200:
    raise ConnectionError(f"Users status code: {users_response.status_code}")
if posts_response.status_code != 200:
    raise ConnectionError(f"Posts status code: {posts_response.status_code}")
users = users_response.json()
posts = posts_response.json()

# (b)
users_by_id = {user["id"]: user for user in users}
joined = {}
for user in users:
    titles = [post["title"] for post in posts if post["userId"] == user["id"]]
    joined[user["name"]] = {"email": user["email"], "post_count": len(titles), "titles": titles}
pprint(dict(list(joined.items())[:3]))

# (c)
top_name, top_data = max(joined.items(), key=lambda item: item[1]["post_count"])
top_user = next(user for user in users if user["name"] == top_name)
print("Name:", top_name)
print("Email:", top_data["email"])
print("Company:", top_user["company"]["name"])
for number, title in enumerate(top_data["titles"], start=1):
    print(f"{number}. {title}")

# (d)
for user in users:
    if "synergies" in user["company"]["bs"].lower() or len(user["address"]["city"]) > 8:
        print(user["name"], user["address"]["city"], joined[user["name"]]["post_count"])

## Task 8: To-Do Completion Rate Analysis Across Users

In [ ]:
# (a)
import requests
from collections import defaultdict

try:
    response = requests.get("https://jsonplaceholder.typicode.com/todos", timeout=10)
    if response.status_code != 200:
        raise ValueError(f"Status code: {response.status_code}")
    todos = response.json()
    if not isinstance(todos, list):
        raise TypeError("The response is not a list.")
    if not todos:
        raise ValueError("No todo records were returned.")
except (requests.RequestException, ValueError, TypeError) as error:
    print("Error:", error)
    raise SystemExit
print("Total todo records:", len(todos))

# (b)
todo_stats = defaultdict(lambda: {"total": 0, "completed": 0, "incomplete": 0})
for todo in todos:
    stats = todo_stats[todo["userId"]]
    stats["total"] += 1
    stats["completed" if todo["completed"] else "incomplete"] += 1
for user_id, stats in sorted(todo_stats.items()):
    print(user_id, stats, stats["completed"] + stats["incomplete"] == stats["total"])

# (c)
rates = {user_id: round(stats["completed"] / stats["total"] * 100, 1) for user_id, stats in todo_stats.items()}
highest = max(rates, key=rates.get)
lowest = min(rates, key=rates.get)
print("Highest completion rate:", highest, rates[highest])
print("Lowest completion rate:", lowest, rates[lowest])
print("Overall average completion rate:", round(sum(rates.values()) / len(rates), 1))

# (d)
users_response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10)
if users_response.status_code != 200:
    raise ConnectionError(f"Users status code: {users_response.status_code}")
user_names = {user["id"]: user["name"] for user in users_response.json()}
print(f"{'Rank':<6}{'Name':<25}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %':<18}")
for rank, (user_id, rate) in enumerate(sorted(rates.items(), key=lambda item: item[1], reverse=True), start=1):
    stats = todo_stats[user_id]
    print(f"{rank:<6}{user_names[user_id]:<25}{stats['total']:<14}{stats['completed']:<12}{rate:<18.1f}")

## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [ ]:
# (a)
import requests
import pandas as pd
import math
from pprint import pprint
from collections import Counter

users_response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10)
if users_response.status_code != 200:
    raise ConnectionError(f"Users status code: {users_response.status_code}")
users = users_response.json()

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

flattened_users = [flatten_user(user) for user in users]
pprint(flattened_users)

# (b)
df = pd.DataFrame(flattened_users)
df[["lat", "lng"]] = df[["lat", "lng"]].astype(float)
df["distance_from_origin"] = df.apply(lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1)
print(df.to_string(index=False))
print("Greatest distance from origin:")
print(df.loc[df["distance_from_origin"].idxmax()])

# (c)
north = df[df["lat"] > 0]
south = df[df["lat"] < 0]
print("Northern Hemisphere:", north["name"].tolist())
print("Southern Hemisphere:", south["name"].tolist())
north_average = north["distance_from_origin"].mean()
south_average = south["distance_from_origin"].mean()
print("Higher average distance:", "Northern Hemisphere" if north_average > south_average else "Southern Hemisphere")

# (d)
posts_response = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=10)
comments_response = requests.get("https://jsonplaceholder.typicode.com/comments", timeout=10)
if posts_response.status_code != 200 or comments_response.status_code != 200:
    raise ConnectionError(f"Posts status: {posts_response.status_code}, comments status: {comments_response.status_code}")
posts = posts_response.json()
comments = comments_response.json()
comment_counts = Counter(comment["postId"] for comment in comments)
post_report = sorted([(post["id"], post["title"][:40], comment_counts[post["id"]]) for post in posts], key=lambda item: item[2], reverse=True)
print("Top 5 most-commented posts:")
for item in post_report[:5]:
    print(item)
print("Bottom 5 least-commented posts:")
for item in post_report[-5:]:
    print(item)